Cargar Modelo Puro 

In [ ]:
from rknn.models_efis import ClassifyModel_EfficientNet
from torchvision.transforms import v2
import torch
from PIL import Image

model = ClassifyModel_EfficientNet(n_class=6)
model.load_state_dict(torch.load("model_EfficientNet.th", map_location="cpu"))  # Cargar pesos entrenados

transform = v2.Compose([
    v2.Resize([224, 224]),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

img_path = "sample_image.jpg"
img = Image.open(img_path).convert("RGB")
img = transform(img).unsqueeze(0)  # Preprocesar imagen
print(img.shape)

model.eval()
with torch.no_grad():
    output = model(img)
    print(output)

Pasar a onnx

In [ ]:

torch.onnx.export(
    model,
    img,
    "model.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["images"],
    output_names=["output"],
    dynamic_axes=None  # ❌ Ningún eje dinámico, entrada estática
)



Probar Onnx

In [ ]:
import onnxruntime

# Crear sesión de ONNX Runtime
ort_session = onnxruntime.InferenceSession("model.onnx")


# Preparar entrada
inputs = {ort_session.get_inputs()[0].name: img.numpy()}


# Ejecutar inferencia
outputs = ort_session.run(None, inputs)

print(outputs[0])

In [ ]:
import onnxruntime as ort

# Cargar modelo
ort_session = ort.InferenceSession("model.onnx")

# Mostrar nombres de inputs
for input_meta in ort_session.get_inputs():
    print("Nombre:", input_meta.name)
    print("Tipo de dato:", input_meta.type)
    print("Dimensiones:", input_meta.shape)


In [ ]:
from rknn.api import RKNN

model_name = "model.onnx"
# Crear objeto RKNN
rknn = RKNN()

rknn = RKNN(verbose=False)
rknn.config(
    #mean_values=[[0.485*255, 0.456*255, 0.406*255]],
    #std_values=[[0.229*255, 0.224*255, 0.225*255]],
    target_platform="rk3588",
    optimization_level=0,
    quantized_dtype="w16a16i_dfp"
)
rknn.load_onnx(model=model_name)
rknn.build(do_quantization=False)
rknn.export_rknn(f"{model_name.split('.')[0]}.rknn")